# 03: Covariates, model fitting, and prediction

This notebook completes the pipeline. Environmental covariates are downloaded from WorldClim, extracted at each presence and background point, and used to fit two classifiers: logistic regression as a simple baseline and a multi-layer perceptron as the neural network comparator.

Both models are fitted under both pseudo-absence strategies from notebook 02. The four resulting predictions are mapped side by side. The aim is to make the effect of methodological choices visible in the final output.

In [ ]:
import os
import sqlite3
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.linear_model    import LogisticRegression
from sklearn.neural_network  import MLPClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics         import roc_auc_score
from sklearn.pipeline        import Pipeline


In [ ]:
with sqlite3.connect('data/gbif_records.db') as conn:
    presences = pd.read_sql('SELECT * FROM presences_thinned',  conn)
    bg_bbox   = pd.read_sql('SELECT * FROM background_bbox',    conn)
    bg_chull  = pd.read_sql('SELECT * FROM background_chull',   conn)

print(f'Presences:        {len(presences):,}')
print(f'Background (bbox):  {len(bg_bbox):,}')
print(f'Background (chull): {len(bg_chull):,}')


## 1. Download WorldClim covariates

WorldClim provides global bioclimatic variables derived from temperature and precipitation records. Four variables are used here:

- BIO1: Annual mean temperature
- BIO4: Temperature seasonality
- BIO12: Annual precipitation
- BIO15: Precipitation seasonality

Data are downloaded at 10 arcmin resolution (~18km). The zip is ~8MB and is skipped on subsequent runs if already present. The raster files are gitignored.

In [ ]:
WCLIM_DIR = 'data/worldclim'
WCLIM_ZIP = os.path.join(WCLIM_DIR, 'wc2.1_10m_bio.zip')
WCLIM_URL = 'https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_bio.zip'

os.makedirs(WCLIM_DIR, exist_ok=True)

if not os.path.exists(WCLIM_ZIP):
    print('Downloading WorldClim (this may take a minute)...')
    urllib.request.urlretrieve(WCLIM_URL, WCLIM_ZIP)
    print('Downloaded.')
else:
    print('Already downloaded.')

if not os.path.exists(os.path.join(WCLIM_DIR, 'wc2.1_10m_bio_1.tif')):
    print('Extracting...')
    with zipfile.ZipFile(WCLIM_ZIP, 'r') as z:
        z.extractall(WCLIM_DIR)
    print('Extracted.')
else:
    print('Already extracted.')


In [ ]:
BIO_VARS = {
    'bio1':  os.path.join(WCLIM_DIR, 'wc2.1_10m_bio_1.tif'),
    'bio4':  os.path.join(WCLIM_DIR, 'wc2.1_10m_bio_4.tif'),
    'bio12': os.path.join(WCLIM_DIR, 'wc2.1_10m_bio_12.tif'),
    'bio15': os.path.join(WCLIM_DIR, 'wc2.1_10m_bio_15.tif'),
}

FEATURES = list(BIO_VARS.keys())


## 2. Extract covariates at occurrence points

Each presence and background point is sampled from the rasters. Points that fall on nodata cells (sea, ice) are dropped.

In [ ]:
def extract_covariates(df):
    coords = list(zip(df['decimalLongitude'], df['decimalLatitude']))
    out    = df[['decimalLongitude', 'decimalLatitude', 'presence']].copy()
    for name, path in BIO_VARS.items():
        with rasterio.open(path) as src:
            nodata = src.nodata
            vals   = np.array([v[0] for v in src.sample(coords)], dtype=float)
            if nodata is not None:
                vals[vals == nodata] = np.nan
        out[name] = vals
    return out.dropna(subset=FEATURES)

pres_cov   = extract_covariates(presences)
bbox_cov   = extract_covariates(bg_bbox)
chull_cov  = extract_covariates(bg_chull)

print(f'Presences with covariates: {len(pres_cov):,}')
print(f'BBox background:           {len(bbox_cov):,}')
print(f'Chull background:          {len(chull_cov):,}')


## 3. Build training datasets

Two datasets are assembled, one per background strategy. Each combines the same thinned presences with a different set of background points.

In [ ]:
ds_bbox  = pd.concat([pres_cov, bbox_cov],  ignore_index=True)
ds_chull = pd.concat([pres_cov, chull_cov], ignore_index=True)

print('Dataset shapes:')
print(f'  BBox:  {ds_bbox.shape}')
print(f'  Chull: {ds_chull.shape}')


## 4. Fit models

A logistic regression and a multi-layer perceptron are fitted on each dataset. Covariates are standardised inside a pipeline so the scaler is never fitted on test data. AUC-ROC is used to evaluate fit; it is threshold-free and appropriate for presence-background data.

In [ ]:
def make_pipelines():
    lr  = Pipeline([('scaler', StandardScaler()),
                    ('model',  LogisticRegression(max_iter=1000))])
    mlp = Pipeline([('scaler', StandardScaler()),
                    ('model',  MLPClassifier(hidden_layer_sizes=(64, 32),
                                             max_iter=500, random_state=42))])
    return lr, mlp

results = {}

for name, ds in [('bbox', ds_bbox), ('chull', ds_chull)]:
    X = ds[FEATURES].values
    y = ds['presence'].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                               random_state=42, stratify=y)
    for label, pipe in zip(['lr', 'mlp'], make_pipelines()):
        pipe.fit(X_tr, y_tr)
        auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
        key = f'{label}_{name}'
        results[key] = {'pipe': pipe, 'auc': auc}
        print(f'{key:15s}  AUC = {auc:.3f}')


## 5. Predict over Aberdeenshire

A regular grid of points is laid over the study area. Covariates are extracted at each grid point and each model predicts presence probability. Points over sea return nodata from the rasters and are dropped before plotting.

In [ ]:
GRID_RES = 0.05
lons = np.arange(-3.75, -1.70, GRID_RES)
lats = np.arange(56.85, 57.80, GRID_RES)
lon_g, lat_g = np.meshgrid(lons, lats)

grid = pd.DataFrame({
    'decimalLongitude': lon_g.ravel(),
    'decimalLatitude':  lat_g.ravel(),
    'presence':         0,
})

grid_cov = extract_covariates(grid)
print(f'Grid points with valid covariates: {len(grid_cov):,}')


In [ ]:
for key, val in results.items():
    grid_cov[f'prob_{key}'] = val['pipe'].predict_proba(grid_cov[FEATURES].values)[:, 1]

grid_cov.head()


## 6. Map predictions

Each panel shows predicted presence probability across Aberdeenshire. Rows are model type; columns are background strategy. The AUC score for each combination is shown in the panel title.

In [ ]:
combos = [
    ('lr_bbox',   'Logistic regression\nBBox background'),
    ('lr_chull',  'Logistic regression\nConvex hull background'),
    ('mlp_bbox',  'MLP\nBBox background'),
    ('mlp_chull', 'MLP\nConvex hull background'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 10), facecolor='#1e1f21')
axes = axes.ravel()

for ax, (key, title) in zip(axes, combos):
    ax.set_facecolor('#1e1f21')
    ax.tick_params(colors='#aaa')
    for spine in ax.spines.values():
        spine.set_edgecolor('#555')
    ax.set_xlim(-3.8, -1.6)
    ax.set_ylim(56.75, 57.85)
    ax.set_xlabel('Longitude', color='#aaa')
    ax.set_ylabel('Latitude',  color='#aaa')

    sc = ax.scatter(
        grid_cov['decimalLongitude'],
        grid_cov['decimalLatitude'],
        c=grid_cov[f'prob_{key}'],
        cmap='RdYlGn', vmin=0, vmax=1,
        s=18, linewidths=0,
    )
    auc = results[key]['auc']
    ax.set_title(f'{title}  (AUC {auc:.3f})', color='white', pad=8, fontsize=10)

fig.colorbar(sc, ax=axes, fraction=0.02, pad=0.02, label='Predicted presence probability')
fig.suptitle('Red squirrel predicted distribution under four model configurations',
             color='white', y=1.01, fontsize=12)
plt.savefig('figures/03_predictions.png', dpi=150,
            bbox_inches='tight', facecolor='#1e1f21')
plt.show()


## What the predictions show

The logistic regression and MLP panels will generally agree on broad patterns but differ at range margins, where the MLP is more sensitive to complex covariate interactions. The more telling contrast is between the bbox and convex hull columns: because the convex hull background excludes coastal margins and high moorland, the model trained against it has a harder discrimination problem and tends to produce tighter, more confident predictions.

AUC scores above 0.7 indicate the models are learning something real. Scores close to 0.5 would mean the background is indistinguishable from presence environments, which is itself a finding: it would suggest the pseudo-absences were drawn from the same environmental space as the presences.

The pipeline from raw GBIF records to a mapped prediction is now complete. Notebook 04 examines what happens to these predictions when the sampling bias in the original records is explicitly corrected.